In [ ]:

import copy
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==================================================
# 1. Experiment configuration
# ==================================================

SEED = 42
DIM = 4
N_PATCHES = 4
BATCH_SIZE = 16
EPOCHS = 40
LR = 0.002
EMA_MOMENTUM = 0.99

OUTPUT_DIR = Path("jepa_results")
OUTPUT_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

DEVICE = "cpu"

# ===================================================
# 2. Synthetic dataset
# ===================================================

def make_images(n, seed):
    generator = torch.Generator().manual_seed(seed)

    a = torch.rand(n, 1, 1, generator=generator) * 0.6 + 0.2
    b = torch.rand(n, 1, 1, generator=generator) * 0.6 + 0.2

    x = torch.linspace(0, 1, 8).view(1, 1, 8)
    y = torch.linspace(0, 1, 8).view(1, 8, 1)

    images = a * x + b * y
    return images.unsqueeze(1).float()


train_images = make_images(256, seed=42)
test_images = make_images(128, seed=123)

def patchify(images):
    return F.unfold(
        images, kernel_size=4, stride=4
    ).transpose(1, 2)

# ==================================================
# 3. Context and target encoder
# ==================================================

class TinyEncoder(nn.Module):
    def __init__(self, dim=4):
        super().__init__()

        self.patch_embed = nn.Linear(16, dim)
        self.pos_embed = nn.Parameter(
            torch.randn(1, N_PATCHES, dim) * 0.02
        )

        layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=2,
            dim_feedforward=16,
            dropout=0.0,
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=1,
            enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(dim)

    def forward(self, patches, indices):
        selected = patches[:, indices, :]

        tokens = (
            self.patch_embed(selected)
            + self.pos_embed[:, indices, :]
        )

        return self.norm(self.transformer(tokens))


# ==================================================
# 4. Classical predictor
# ==================================================

class ClassicalPredictor(nn.Module):
    def __init__(self, dim=4):
        super().__init__()

        self.target_pos = nn.Embedding(N_PATCHES, dim)

        self.network = nn.Sequential(
            nn.Linear(dim * 2, 16),
            nn.GELU(),
            nn.Linear(16, dim)
        )

    def forward(self, context, target_idx):

        summary = context.mean(dim=1)

        positions = torch.full(
            (context.shape[0],),
            target_idx,
            dtype=torch.long,
            device=context.device
        )

        pos = self.target_pos(positions)

        return self.network(
            torch.cat([summary, pos], dim=-1)
        )


# ==================================================
# 5. Initialize models
# ==================================================

encoder = TinyEncoder(DIM).to(DEVICE)
target_encoder = copy.deepcopy(encoder)
predictor = ClassicalPredictor(DIM).to(DEVICE)

for p in target_encoder.parameters():
    p.requires_grad_(False)

# Save initial states for later controlled
# classical-versus-quantum comparisons.
initial_state = {
    "seed": SEED,
    "encoder": copy.deepcopy(encoder.state_dict()),
    "target_encoder": copy.deepcopy(
        target_encoder.state_dict()
    ),
    "predictor": copy.deepcopy(predictor.state_dict())
}

torch.save(
    initial_state,
    OUTPUT_DIR / "initial_state.pt"
)

optimizer = torch.optim.Adam(
    list(encoder.parameters())
    + list(predictor.parameters()),
    lr=LR
)

# ==================================================
# 6. Evaluation
# ==================================================

@torch.no_grad()
def collect_predictions(images):
    encoder.eval()
    target_encoder.eval()
    predictor.eval()

    all_predictions = []
    all_targets = []

    # Evaluate every possible masked-patch position
    for start in range(0, len(images), BATCH_SIZE):
        batch = images[start:start + BATCH_SIZE]
        patches = patchify(batch)

        target_tokens = target_encoder(
            patches, list(range(N_PATCHES))
        )

        for target_idx in range(N_PATCHES):
            context_idx = [
                i for i in range(N_PATCHES)
                if i != target_idx
            ]

            context = encoder(patches, context_idx)
            prediction = predictor(context, target_idx)
            target = target_tokens[:, target_idx, :]

            all_predictions.append(prediction)
            all_targets.append(target)

    return (
        torch.cat(all_predictions, dim=0),
        torch.cat(all_targets, dim=0)
    )


def evaluate(images, constant=None):
    predictions, targets = collect_predictions(images)

    mse = F.mse_loss(predictions, targets).item()
    smooth = F.smooth_l1_loss(
        predictions, targets
    ).item()

    target_var = targets.var(
        dim=0, unbiased=False
    ).mean().item()

    prediction_var = predictions.var(
        dim=0, unbiased=False
    ).mean().item()

    result = {
        "mse": mse,
        "smooth_l1": smooth,
        "target_variance": target_var,
        "prediction_variance": prediction_var
    }

    if constant is not None:
        const_prediction = constant.expand_as(targets)

        result["constant_mse"] = F.mse_loss(
            const_prediction, targets
        ).item()

    return result


# ==================================================
# 7. Classical JEPA training
# ==================================================

print("\nStarting classical JEPA validation")
print("Training examples:", len(train_images))
print("Test examples:", len(test_images))

history = []

for epoch in range(EPOCHS):

    encoder.train()
    predictor.train()
    target_encoder.eval()

    indices = torch.randperm(len(train_images))
    total_loss = 0.0

    for start in range(0, len(indices), BATCH_SIZE):

        batch = train_images[
            indices[start:start + BATCH_SIZE]
        ]

        patches = patchify(batch)

        target_idx = torch.randint(
            0, N_PATCHES, (1,)
        ).item()

        context_idx = [
            i for i in range(N_PATCHES)
            if i != target_idx
        ]

        context = encoder(patches, context_idx)

        with torch.no_grad():
            target_tokens = target_encoder(
                patches, list(range(N_PATCHES))
            )
            target = target_tokens[:, target_idx, :]

        prediction = predictor(context, target_idx)

        loss = F.smooth_l1_loss(
            prediction, target
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Update EMA target encoder
        with torch.no_grad():
            for p, tp in zip(
                encoder.parameters(),
                target_encoder.parameters()
            ):
                tp.mul_(EMA_MOMENTUM).add_(
                    p, alpha=1 - EMA_MOMENTUM
                )

        total_loss += loss.item() * len(batch)

    train_epoch_loss = total_loss / len(train_images)

    train_eval = evaluate(train_images)
    test_eval = evaluate(test_images)

    history.append({
        "epoch": epoch,
        "train_mse": train_eval["mse"],
        "test_mse": test_eval["mse"],
        "target_variance": test_eval["target_variance"],
        "prediction_variance": test_eval[
            "prediction_variance"
        ]
    })

    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(
            f"Epoch {epoch:02d} | "
            f"Train MSE: {train_eval['mse']:.6f} | "
            f"Test MSE: {test_eval['mse']:.6f} | "
            f"Target Var: {test_eval['target_variance']:.6f}"
        )

# ==================================================
# 8. Final diagnostics
# ==================================================

train_predictions, train_targets = collect_predictions(
    train_images
)

# Mean target representation from training data only
constant_vector = train_targets.mean(
    dim=0, keepdim=True
)

train_results = evaluate(
    train_images, constant=constant_vector
)

test_results = evaluate(
    test_images, constant=constant_vector
)

print("\n" + "=" * 50)
print("FINAL CLASSICAL JEPA DIAGNOSTICS")
print("=" * 50)

for name, results in [
    ("TRAIN", train_results),
    ("TEST", test_results)
]:
    print(f"\n{name}")
    for metric, value in results.items():
        print(f"{metric:22s}: {value:.8f}")

# ==================================================
# 9. Save trained checkpoint and results
# ==================================================

torch.save({
    "seed": SEED,
    "config": {
        "dim": DIM,
        "n_patches": N_PATCHES,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "ema_momentum": EMA_MOMENTUM
    },
    "encoder": encoder.state_dict(),
    "target_encoder": target_encoder.state_dict(),
    "predictor": predictor.state_dict(),
    "optimizer": optimizer.state_dict(),
    "history": history,
    "train_results": train_results,
    "test_results": test_results,
    "constant_vector": constant_vector
}, OUTPUT_DIR / "classical_jepa.pt")

print("\nSaved files:")
print("jepa_results/initial_state.pt")
print("jepa_results/classical_jepa.pt")

print("\nExperiment completed.")



Starting classical JEPA validation
Training examples: 256
Test examples: 128
Epoch 00 | Train MSE: 0.477487 | Test MSE: 0.475246 | Target Var: 0.025018
Epoch 05 | Train MSE: 0.030405 | Test MSE: 0.031619 | Target Var: 0.023059
Epoch 10 | Train MSE: 0.008311 | Test MSE: 0.008360 | Target Var: 0.008134
Epoch 15 | Train MSE: 0.004765 | Test MSE: 0.004585 | Target Var: 0.007758
Epoch 20 | Train MSE: 0.005133 | Test MSE: 0.004743 | Target Var: 0.012206
Epoch 25 | Train MSE: 0.006171 | Test MSE: 0.005917 | Target Var: 0.026395
Epoch 30 | Train MSE: 0.015258 | Test MSE: 0.014741 | Target Var: 0.057764
Epoch 35 | Train MSE: 0.031014 | Test MSE: 0.029415 | Target Var: 0.116657
Epoch 39 | Train MSE: 0.048827 | Test MSE: 0.048455 | Target Var: 0.176672

FINAL CLASSICAL JEPA DIAGNOSTICS

TRAIN
mse                   : 0.04882747
smooth_l1             : 0.02441374
target_variance       : 0.19780457
prediction_variance   : 0.09312674
constant_mse          : 0.19780457

TEST
mse                   : 0

In [2]:
@torch.no_grad()
def per_position_variance(images):
    target_encoder.eval()
    patches = patchify(images)

    targets = target_encoder(
        patches, list(range(N_PATCHES))
    )

    print("\nVariance across images, by patch position:")

    for i in range(N_PATCHES):
        variance = targets[:, i, :].var(
            dim=0, unbiased=False
        ).mean().item()

        print(f"Patch {i}: {variance:.8f}")

per_position_variance(test_images)


Variance across images, by patch position:
Patch 0: 0.17058232
Patch 1: 0.15695006
Patch 2: 0.16338934
Patch 3: 0.12044155


In [3]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

torch.manual_seed(42)
torch.set_num_threads(2)

DIM = 4
DEVICE = "cpu"

# -------------------------------------------
# 1. Reproduce synthetic data with labels
# -------------------------------------------

def make_images_and_labels(n, seed):
    generator = torch.Generator().manual_seed(seed)

    a = torch.rand(n, 1, 1, generator=generator) * 0.6 + 0.2
    b = torch.rand(n, 1, 1, generator=generator) * 0.6 + 0.2

    x = torch.linspace(0, 1, 8).view(1, 1, 8)
    y = torch.linspace(0, 1, 8).view(1, 8, 1)

    images = (a * x + b * y).unsqueeze(1).float()
    labels = torch.cat(
        [a.reshape(n, 1), b.reshape(n, 1)], dim=1
    )

    return images, labels


train_images, train_labels = make_images_and_labels(256, 42)
test_images, test_labels = make_images_and_labels(128, 123)

# -------------------------------------------
# 2. Reconstruct original JEPA encoder
# -------------------------------------------

class TinyEncoder(nn.Module):
    def __init__(self, dim=4):
        super().__init__()

        self.patch_embed = nn.Linear(16, dim)
        self.pos_embed = nn.Parameter(
            torch.randn(1, 4, dim) * 0.02
        )

        layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=2,
            dim_feedforward=16,
            dropout=0.0,
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=1,
            enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(dim)

    def forward(self, patches, indices):
        selected = patches[:, indices, :]

        tokens = (
            self.patch_embed(selected)
            + self.pos_embed[:, indices, :]
        )

        return self.norm(self.transformer(tokens))


def patchify(images):
    return F.unfold(
        images, kernel_size=4, stride=4
    ).transpose(1, 2)

# -------------------------------------------
# 3. Load frozen JEPA target encoder
# -------------------------------------------

checkpoint_path = Path(
    "jepa_results/classical_jepa.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

encoder = TinyEncoder(DIM)
encoder.load_state_dict(
    checkpoint["target_encoder"]
)
encoder.eval()

for parameter in encoder.parameters():
    parameter.requires_grad_(False)

print("Trained JEPA target encoder loaded.")

# -------------------------------------------
# 4. Extract image-level representations
# -------------------------------------------

@torch.no_grad()
def extract_features(images):
    patches = patchify(images)

    tokens = encoder(
        patches, [0, 1, 2, 3]
    )

    return tokens.mean(dim=1)


X_train = extract_features(train_images)
X_test = extract_features(test_images)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)

# -------------------------------------------
# 5. Train linear probe
# -------------------------------------------

probe = nn.Linear(DIM, 2)

optimizer = torch.optim.Adam(
    probe.parameters(),
    lr=0.01
)

for epoch in range(501):
    optimizer.zero_grad()

    predictions = probe(X_train)
    loss = F.mse_loss(
        predictions, train_labels
    )

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Probe train MSE: {loss.item():.8f}"
        )

# -------------------------------------------
# 6. Evaluate held-out performance
# -------------------------------------------

@torch.no_grad()
def evaluate(features, labels):
    predictions = probe(features)

    mse = F.mse_loss(
        predictions, labels
    ).item()

    mae = F.l1_loss(
        predictions, labels
    ).item()

    ss_res = (
        (labels - predictions) ** 2
    ).sum(dim=0)

    ss_tot = (
        (labels - labels.mean(dim=0)) ** 2
    ).sum(dim=0)

    r2 = 1 - ss_res / ss_tot

    return predictions, mse, mae, r2


train_pred, train_mse, _, _ = evaluate(
    X_train, train_labels
)

test_pred, test_mse, test_mae, test_r2 = evaluate(
    X_test, test_labels
)

# A constant baseline uses training-label means
baseline = train_labels.mean(
    dim=0, keepdim=True
).expand_as(test_labels)

baseline_mse = F.mse_loss(
    baseline, test_labels
).item()

print("\n" + "=" * 48)
print("LINEAR PROBE RESULTS")
print("=" * 48)

print(f"Train MSE:         {train_mse:.8f}")
print(f"Test MSE:          {test_mse:.8f}")
print(f"Test MAE:          {test_mae:.8f}")
print(f"Test R2 for a:     {test_r2[0].item():.6f}")
print(f"Test R2 for b:     {test_r2[1].item():.6f}")
print(f"Baseline test MSE: {baseline_mse:.8f}")

print("\nFive held-out examples:")
for i in range(5):
    print(
        f"True: {test_labels[i].tolist()} "
        f"Predicted: {test_pred[i].tolist()}"
    )

# -------------------------------------------
# 7. Save probe separately
# -------------------------------------------

torch.save(
    {
        "probe": probe.state_dict(),
        "train_mse": train_mse,
        "test_mse": test_mse,
        "test_r2": test_r2.tolist(),
        "baseline_mse": baseline_mse,
    },
    "jepa_results/linear_probe.pt"
)

print("\nLinear probe checkpoint saved.")


Trained JEPA target encoder loaded.
Training features: torch.Size([256, 4])
Test features: torch.Size([128, 4])
Epoch 000 | Probe train MSE: 1.81500363
Epoch 100 | Probe train MSE: 0.03049642
Epoch 200 | Probe train MSE: 0.01873410
Epoch 300 | Probe train MSE: 0.01685418
Epoch 400 | Probe train MSE: 0.01662566
Epoch 500 | Probe train MSE: 0.01651501

LINEAR PROBE RESULTS
Train MSE:         0.01651394
Test MSE:          0.01657171
Test MAE:          0.10784039
Test R2 for a:     0.344824
Test R2 for b:     0.492887
Baseline test MSE: 0.02896119

Five held-out examples:
True: [0.3776671886444092, 0.5837984681129456] Predicted: [0.5158703327178955, 0.5253721475601196]
True: [0.5099374055862427, 0.7372409105300903] Predicted: [0.5915092825889587, 0.6338372826576233]
True: [0.35100245475769043, 0.37875068187713623] Predicted: [0.37641122937202454, 0.34630972146987915]
True: [0.6131340861320496, 0.578845739364624] Predicted: [0.5796772241592407, 0.6125082969665527]
True: [0.24438348412513733

In [4]:

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# ==================================================
# 1. Configuration
# ==================================================

SEED = 42
DIM = 4
N_TRAIN = 256
N_TEST = 128

torch.manual_seed(SEED)
torch.set_num_threads(2)

RESULTS = Path("jepa_results")

# ==================================================
# 2. Reproduce the original synthetic dataset
# ==================================================

def make_images_and_labels(n, seed):

    generator = torch.Generator().manual_seed(seed)

    a = torch.rand(
        n, 1, 1, generator=generator
    ) * 0.6 + 0.2

    b = torch.rand(
        n, 1, 1, generator=generator
    ) * 0.6 + 0.2

    x = torch.linspace(0, 1, 8).view(1, 1, 8)
    y = torch.linspace(0, 1, 8).view(1, 8, 1)

    images = (a * x + b * y).unsqueeze(1)

    labels = torch.cat([
        a.reshape(n, 1),
        b.reshape(n, 1)
    ], dim=1)

    return images.float(), labels.numpy()


train_images, y_train = make_images_and_labels(
    N_TRAIN, 42
)

test_images, y_test = make_images_and_labels(
    N_TEST, 123
)

# ==================================================
# 3. Same encoder architecture as I_JEPA_04
# ==================================================

class TinyEncoder(nn.Module):

    def __init__(self, dim=4):
        super().__init__()

        self.patch_embed = nn.Linear(16, dim)

        self.pos_embed = nn.Parameter(
            torch.randn(1, 4, dim) * 0.02
        )

        layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=2,
            dim_feedforward=16,
            dropout=0.0,
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=1,
            enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(dim)

    def forward(self, patches, indices):

        selected = patches[:, indices, :]

        tokens = (
            self.patch_embed(selected)
            + self.pos_embed[:, indices, :]
        )

        return self.norm(self.transformer(tokens))


def patchify(images):
    return F.unfold(
        images, kernel_size=4, stride=4
    ).transpose(1, 2)

# ==================================================
# 4. Load initial and trained encoders
# ==================================================

initial_checkpoint = torch.load(
    RESULTS / "initial_state.pt",
    map_location="cpu",
    weights_only=True
)

trained_checkpoint = torch.load(
    RESULTS / "classical_jepa.pt",
    map_location="cpu",
    weights_only=True
)

random_encoder = TinyEncoder(DIM)
random_encoder.load_state_dict(
    initial_checkpoint["target_encoder"]
)
random_encoder.eval()

trained_encoder = TinyEncoder(DIM)
trained_encoder.load_state_dict(
    trained_checkpoint["target_encoder"]
)
trained_encoder.eval()

for model in (random_encoder, trained_encoder):
    for p in model.parameters():
        p.requires_grad_(False)

print("Initial and trained encoders loaded.")

# ==================================================
# 5. Extract features
# ==================================================

@torch.no_grad()
def encoder_features(model, images):

    patches = patchify(images)

    tokens = model(
        patches, [0, 1, 2, 3]
    )

    # One image-level latent vector
    return tokens.mean(dim=1).numpy()


# A. Raw pixel representation
X_raw_train = train_images.reshape(N_TRAIN, -1).numpy()
X_raw_test = test_images.reshape(N_TEST, -1).numpy()

# B. Randomly initialized JEPA representation
X_random_train = encoder_features(
    random_encoder, train_images
)
X_random_test = encoder_features(
    random_encoder, test_images
)

# C. Trained JEPA representation
X_jepa_train = encoder_features(
    trained_encoder, train_images
)
X_jepa_test = encoder_features(
    trained_encoder, test_images
)

# ==================================================
# 6. Linear probing
# ==================================================

def evaluate_representation(name, X_train, X_test):

    probe = LinearRegression()

    probe.fit(X_train, y_train)

    train_prediction = probe.predict(X_train)
    test_prediction = probe.predict(X_test)

    train_mse = mean_squared_error(
        y_train, train_prediction
    )

    test_mse = mean_squared_error(
        y_test, test_prediction
    )

    r2 = r2_score(
        y_test,
        test_prediction,
        multioutput="raw_values"
    )

    print(f"\n{name}")
    print("-" * 45)
    print(f"Feature dimensions: {X_train.shape[1]}")
    print(f"Train MSE: {train_mse:.8f}")
    print(f"Test MSE:  {test_mse:.8f}")
    print(f"R2(a):     {r2[0]:.6f}")
    print(f"R2(b):     {r2[1]:.6f}")

    return {
        "name": name,
        "train_mse": train_mse,
        "test_mse": test_mse,
        "r2_a": r2[0],
        "r2_b": r2[1]
    }


print("\n" + "=" * 55)
print("EXPERIMENT 6: REPRESENTATION BENCHMARK")
print("=" * 55)

results = []

results.append(evaluate_representation(
    "A - Raw images",
    X_raw_train, X_raw_test
))

results.append(evaluate_representation(
    "B - Random encoder",
    X_random_train, X_random_test
))

results.append(evaluate_representation(
    "C - Trained JEPA encoder",
    X_jepa_train, X_jepa_test
))

# ==================================================
# 7. Summary
# ==================================================

print("\n" + "=" * 55)
print("FINAL COMPARISON")
print("=" * 55)

print(
    f"{'Model':25s} "
    f"{'Test MSE':>10s} "
    f"{'R2(a)':>10s} "
    f"{'R2(b)':>10s}"
)

for r in results:
    print(
        f"{r['name']:25s} "
        f"{r['test_mse']:10.6f} "
        f"{r['r2_a']:10.4f} "
        f"{r['r2_b']:10.4f}"
    )

print("\nExperiment 6 completed.")



Initial and trained encoders loaded.

EXPERIMENT 6: REPRESENTATION BENCHMARK

A - Raw images
---------------------------------------------
Feature dimensions: 64
Train MSE: 0.00000000
Test MSE:  0.00000000
R2(a):     1.000000
R2(b):     1.000000

B - Random encoder
---------------------------------------------
Feature dimensions: 4
Train MSE: 0.00002050
Test MSE:  0.00002059
R2(a):     0.999516
R2(b):     0.999036

C - Trained JEPA encoder
---------------------------------------------
Feature dimensions: 4
Train MSE: 0.01090029
Test MSE:  0.01180267
R2(a):     0.541673
R2(b):     0.630413

FINAL COMPARISON
Model                       Test MSE      R2(a)      R2(b)
A - Raw images              0.000000     1.0000     1.0000
B - Random encoder          0.000021     0.9995     0.9990
C - Trained JEPA encoder    0.011803     0.5417     0.6304

Experiment 6 completed.
